In [1]:
import pandas as pd

In [2]:
products = pd.read_csv('../dataset/produtos_clean.csv')
sales = pd.read_csv('../dataset/vendas_2023_2024.csv')

In [ ]:
sales_enriched = sales.merge(
    products,
    left_on='id_product',
    right_on='code'
)

agg_by_client = (
    sales_enriched.groupby('id_client', as_index=False)
    .agg(
        categories_count = ('actual_category', 'nunique'),
        purchase_count = ('id', 'size'),
        total_revenue = ('total', 'sum')
    )
    
)

agg_by_client['avg_spent_value'] = agg_by_client['total_revenue'] / agg_by_client['purchase_count']

agg_by_client.head()

,id_client,categories_count,purchase_count,total_revenue,avg_spent_value
0,1,3,190,51092500.05,268907.895000
1,2,3,220,65652931.35,298422.415227
2,3,3,207,59575349.10,287803.618841
3,4,3,207,50691754.40,244887.702415
4,5,3,202,58592802.70,290063.379703


In [ ]:
top_10_clients = (
    agg_by_client[agg_by_client['categories_count'] >= 3]
    .sort_values(['avg_spent_value', 'id_client'], ascending=[False, True])
    .head(10)
)

top_10_clients

,id_client,categories_count,purchase_count,total_revenue,avg_spent_value
46,47,3,190,64003343.75,336859.703947
41,42,3,222,72187369.50,325168.331081
8,9,3,218,66788855.35,306370.896101
21,22,3,198,59581398.75,300916.155303
1,2,3,220,65652931.35,298422.415227
27,28,3,204,60826837.25,298170.770833
45,46,3,199,59126834.35,297119.770603
37,38,3,195,57093331.15,292786.313590
35,36,3,215,62791038.15,292051.340233
4,5,3,202,58592802.70,290063.379703


In [ ]:
most_bought_category = (
    sales_enriched[sales_enriched['id_client'].isin(top_10_clients['id_client'])]
    .groupby('actual_category')['qtd']
    .sum()
    .sort_values(ascending=False)
    .head(1)
)

most_bought_category

actual_category
propulsão    6030
Name: qtd, dtype: int64